# 03 — Stimulus and response

**Which frames belong to which component?**

The method: record with `bikelog`, mark the instant you operate something
(`headlight on`, Enter), then ask what traffic appeared inside that window that
had never appeared before it.

A mark is typed by a person, so the thing being marked usually happens a moment
*before* the Enter key. `experiment.baseline` leaves a one-second guard for
exactly this.

In [ ]:
import sys
from pathlib import Path

# The project is not installed as a package, so put the repo root on the path.
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO))

import pandas as pd
pd.set_option("display.width", 200, "display.max_columns", 40)

from bikecan import dbc, discover, experiment, ids, session

SESSIONS = REPO / "data" / "sessions"

In [ ]:
# Point this at a session recorded with marks.
# `bikelog list` on the Pi shows what is available; tools/pull-session.sh pulls it.
candidates = [d for d in sorted(SESSIONS.glob("*")) if d.is_dir() and d.name != "legacy"]
print("sessions available:")
for d in candidates:
    print(" ", d.name)

s = session.load(candidates[-1]) if candidates else session.load(SESSIONS / "legacy")
print()
print(s.summary())

## The windows this session contains

Two marks with the same label bracket a window; a lone mark opens a
five-second one.

In [ ]:
windows = experiment.windows(s.marks)
for window in windows:
    print(window)
if not windows:
    print("no marks in this session -- there is nothing to compare against")

## What was different inside each window

`new_id` is the strong result: an identifier that appears only while a component
is being operated belongs to that component. `new_payloads` is the weaker one —
a message that was already there but started saying something new.

In [ ]:
print(experiment.report(s.can, s.marks))

## One window in detail

In [ ]:
if windows:
    window = windows[0]
    inside = window.slice(s.can)
    print(f"{window}\n{len(inside)} frames inside the window\n")
    display(experiment.diff_window(s.can, window))
    display(inside.groupby("id_hex").agg(
        frames=("id_hex", "size"),
        payloads=("data_hex", "nunique"),
        first=("data_hex", "first"),
        last=("data_hex", "last"),
    ))

## Serial correlation

A cellular command arriving on the serial console (unlock, immobilise, lock)
should be followed by CAN traffic. Anchoring on serial events finds the command
frames that no mechanical bench test can trigger.

In [ ]:
if len(s.serial):
    from bikecan import serialreader
    hits = serialreader.grep(s.serial, r"unlock|lock|immobil|command|cmd")
    display(hits[["timestamp", "text"]].head(20))

    for _, line in hits.head(5).iterrows():
        window = experiment.Window(line["text"][:40], line["timestamp"], line["timestamp"] + 2.0)
        diff = experiment.diff_window(s.can, window)
        interesting = diff[diff["interesting"]] if len(diff) else diff
        print(f"\n{window}")
        print(interesting.to_string(index=False) if len(interesting) else "  nothing new")
else:
    print("no serial lines in this session")

## Write down what you found

Anything you conclude goes in `dbc/nodes.md` (which component an address is)
or `dbc/signals.toml` (what a field means), **with the evidence and a confidence
level**. Then regenerate the DBC:

```
uv run python -m bikecan.dbc
```